# Databricks Fundamentals Examples

This notebook demonstrates a few common Databricks patterns:

- notebook parameters with widgets
- Spark configuration tuning for a small workload
- DataFrame transformations and aggregations
- writing results to a Delta table

In [ ]:
from pyspark.sql import functions as F

In [ ]:
# Databricks widgets make notebooks reusable in jobs and manual runs.
dbutils.widgets.text("load_date", "2026-04-01", "Load Date")
dbutils.widgets.dropdown("run_mode", "incremental", ["incremental", "full"], "Run Mode")

load_date = dbutils.widgets.get("load_date")
run_mode = dbutils.widgets.get("run_mode")

print(f"load_date={load_date}, run_mode={run_mode}")

In [ ]:
# Tune a few Spark settings for a small demo workload.
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

In [ ]:
orders = [
    (1, "Laptop", "electronics", 1200.0, "2026-04-01"),
    (2, "Mouse", "electronics", 25.0, "2026-04-01"),
    (3, "Desk", "furniture", 300.0, "2026-04-01"),
    (4, "Chair", "furniture", 150.0, "2026-04-02"),
    (5, "Monitor", "electronics", 400.0, "2026-04-02")
]

orders_df = spark.createDataFrame(
    orders,
    ["order_id", "product", "category", "amount", "order_date"]
)

display(orders_df)

In [ ]:
filtered_df = orders_df.filter(F.col("order_date") >= load_date)

summary_df = (
    filtered_df
    .groupBy("category")
    .agg(
        F.count("*").alias("order_count"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("avg_amount")
    )
    .withColumn("load_date", F.lit(load_date))
)

display(summary_df.orderBy("category"))

In [ ]:
target_table = "main.demo.category_sales_summary"

summary_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
print(f"Wrote summary to {target_table}")

In [ ]:
result_df = spark.sql("""
SELECT category, order_count, total_amount, avg_amount, load_date
FROM main.demo.category_sales_summary
ORDER BY total_amount DESC
""")

display(result_df)

## How this notebook maps to production

- Convert widget values into job parameters
- Replace inline sample data with cloud storage or catalog tables
- Write to a curated schema in Unity Catalog
- Run the notebook from a scheduled job or multi-task workflow